# Your first PSF

`physicaloptix` turns a description of an optical system into point-spread
functions by wave-optics propagation. This notebook builds up the smallest
complete example: a wavefront on a pupil grid, propagated to a focal plane to
form an Airy pattern, and then a two-stage coronagraph that nulls an on-axis
star while passing an off-axis planet.

Everything here uses only the owned propagation core: the `Grid` / `Field` /
`PlaneKind` data model, the `Fraunhofer` propagator, and the `OpticalPath`
fold. No hardware model, no scene, no detector -- just the diffraction.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)  # deep contrast floors need float64

import equinox as eqx
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm

import physicaloptix as po
from physicaloptix import (
    Field,
    Fraunhofer,
    Grid,
    MultiScaleVortex,
    OpticalPath,
    PlaneKind,
    Stage,
)
from physicaloptix.elements import SampledOptic

print("physicaloptix", po.__version__)

## The core data model: `Grid`, `Field`, `PlaneKind`

A `Grid` is an all-static sampling of a plane. `Grid.pupil(n)` gives an
`n`-by-`n` pupil grid spanning the unit aperture; `Grid.focal(n, pixel_scale)`
gives a focal-plane grid sampled at `pixel_scale` in units of $\lambda/D$.

A `Field` is the complex wavefront living on a grid, tagged with the kind of
plane it sits in (`PlaneKind.PUPIL` or `PlaneKind.FOCAL`). The tag is what lets
propagators check, at construction time, that they are being handed the plane
they expect.

Here we make a small pupil grid, a matching oversampled focal grid, and put a
circular clear aperture on the pupil as a unit-amplitude `Field`.

In [ ]:
NPUP, NFOC, PSCALE = 48, 96, 0.2  # toy grids; focal FOV is +-9.6 lambda/D

pupil = Grid.pupil(NPUP)
focal = Grid.focal(NFOC, PSCALE)

x = np.asarray(pupil.coords)
xg, yg = np.meshgrid(x, x)
aperture = ((xg**2 + yg**2) <= 0.25).astype(complex)  # radius 0.5 clear disk

flat = Field(data=jnp.asarray(aperture), grid=pupil, plane=PlaneKind.PUPIL)
print(flat)

## Propagating to a focal plane

The `Fraunhofer` propagator maps a pupil `Field` to a focal `Field` with the
validated continuous-Fourier-transform MFT pair. You do not call it directly:
you wrap it in a `Stage` and fold one or more stages into an `OpticalPath`,
then `propagate` a field through the whole chain. Here the chain is a single
stage -- a bare telescope -- so the output is the Airy pattern of the clear
aperture.

In [ ]:
tele = OpticalPath(stages=(Stage("sci", Fraunhofer(grid_in=pupil, grid_out=focal)),))
airy, _ = tele.propagate(flat)

psf0 = np.abs(np.asarray(airy.data)) ** 2
TELE_PEAK, TELE_SUM = psf0.max(), psf0.sum()
ext = [-NFOC / 2 * PSCALE, NFOC / 2 * PSCALE] * 2

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
axes[0].imshow(np.abs(aperture), origin="lower", cmap="gray")
axes[0].set_title("pupil amplitude")
axes[0].set_xticks([]), axes[0].set_yticks([])
im = axes[1].imshow(
    psf0 / TELE_PEAK,
    origin="lower",
    cmap="inferno",
    norm=LogNorm(1e-7, 1),
    extent=ext,
    interpolation="none",
)
axes[1].set_title("focal intensity (Airy)")
axes[1].set_xlabel(r"$\lambda/D$")
fig.colorbar(im, ax=axes[1], label="contrast")
plt.show()

## Nulling a star: a vortex coronagraph

A coronagraph is just a longer `OpticalPath`. Here we fold three stages:

1. a charge-2 `MultiScaleVortex` focal-plane mask, built on a multi-scale grid
   ladder so the sharp phase ramp at its center is well sampled;
2. a `SampledOptic` Lyot stop in the re-imaged pupil that blocks the light the
   vortex has diffracted to the pupil edge;
3. a final `Fraunhofer` to the science focal plane.

`po.render_path` draws the field at every plane of the path, which is the
fastest way to see what each stage does to an on-axis wavefront.

In [ ]:
lyot = ((xg**2 + yg**2) <= 0.45**2).astype(float)
vortex_train = OpticalPath(
    stages=(
        Stage(
            "vortex",
            MultiScaleVortex.build(
                charge=2, npup=NPUP, q=64, scaling_factor=4, window_size=16
            ),
        ),
        Stage(
            "lyot",
            SampledOptic(
                transmission=jnp.asarray(lyot), grid=pupil, plane=PlaneKind.PUPIL
            ),
        ),
        Stage("sci", Fraunhofer(grid_in=pupil, grid_out=focal)),
    )
)
po.render_path(vortex_train, flat, title="charge-2 vortex train, on axis")
plt.show()

The on-axis star is nulled, but an off-axis source is not. We propagate the
same path twice: once with the flat on-axis wavefront, and once with an added
tilt of 8 $\lambda/D$ standing in for a planet. The star is suppressed by
orders of magnitude; the planet passes through almost untouched.

In [ ]:
out_on, _ = vortex_train.propagate(flat)
null = float(np.sum(np.abs(np.asarray(out_on.data)) ** 2) / TELE_SUM)

tilt8 = np.exp(1j * 2 * np.pi * 8.0 * xg)
off_axis = eqx.tree_at(lambda f: f.data, flat, flat.data * jnp.asarray(tilt8))
out_off, _ = vortex_train.propagate(off_axis)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
titles = (f"on axis: null = {null:.1e} of input", r"off axis (8 $\lambda/D$): passes")
for ax, out, title in zip(axes, (out_on, out_off), titles):
    ax.imshow(
        np.abs(np.asarray(out.data)) ** 2 / TELE_PEAK,
        origin="lower",
        cmap="inferno",
        norm=LogNorm(1e-10, 1),
        extent=ext,
        interpolation="none",
    )
    ax.set_title(title)
    ax.set_xlabel(r"$\lambda/D$")
plt.show()

print(f"perfect-wavefront vortex null: {null:.2e} of the input starlight")

The remaining floor here is set by the coarse toy grid, not by physics. On the
properly sampled grids used in the validation suite, the same charge-2 vortex
reaches an on-axis null of 3.05e-11 against the HWO Coronagraph Design Survey
EAC-1 AAVC reference (0.2 percent of the reference null).

## A realistic aperture

The clear disk above kept the example self-contained, but physicaloptix ships
the real HWO EAC-1 segmented primary, rasterized from the same YAML the mission
config uses. This is the pupil you would actually propagate through.

In [ ]:
from physicaloptix.apertures import eac1_primary, rasterize_primary

prim = eac1_primary()
amp_eac1 = np.asarray(rasterize_primary(prim, 192))

fig, ax = plt.subplots(figsize=(4.4, 4.2))
ax.imshow(amp_eac1, origin="lower", cmap="gray")
ax.set_title(
    f"EAC-1: {prim.n_segments} segments, {prim.diameter_m:.2f} m circumscribed"
)
ax.set_xticks([]), ax.set_yticks([])
plt.show()

## Where to go next

- **`PathCoronagraph`** wraps an `OpticalPath` behind optixstuff's
  `AbstractCoronagraph`, deriving its IWA and throughput curves from the
  propagated PSFs, so any downstream tool (coronagraphoto, jaxedith) can
  consume it without touching diffraction code.
- **Elements** beyond the vortex and Lyot -- `ModeBasis`, `PhaseScreen`,
  Zernike and deformable-mirror bases -- let you inject and correct wavefront
  error.
- **The speckle layer** (`linearize`, `SpeckleProcess`,
  `AnalyticSpeckleField`) turns a path into the linear `(E_nom, G)` speckle
  model that feeds `physicaloptix.stats`; [notebook 6](06_The_Speckle_Layer_in_Code)
  drives it end to end.

For a gentle, ordered build-up of these ideas, work through the tutorials starting with [Grids, Fields, and Planes](01_Grids_and_Fields). See the API reference for the full surface.